In [ ]:
!pip install transformers datasets huggingface_hub


# Transformers Introduction TP

In this notebook, we are going to explore Hugging Face Transformers with several tasks:
- Text Classification  
- Named Entity Recognition (NER)  
- Question Answering  
- Summarization  
- Translation  
- Text Generation


In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""


## Question 1: Understanding Pipelines

**What is a pipeline?**  
Tokenizers, models, and post-processing steps are all combined into a high-level abstraction called a pipeline.  All low-level operations (decoding, tensors, tokenization) are concealed.

**Others pipeline tasks:**  
- question-answering  
- summarization  
- translation  
- text-generation  

**What's happening if there is no model specified?**  
Transformers chooses a default model for the task

**How to specify a model?**  
Example to:  
`pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")`


In [ ]:
from transformers import pipeline
import pandas as pd

classifier = pipeline("text-classification")
outputs = classifier(text)
pd.DataFrame(outputs)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


,label,score
0,NEGATIVE,0.901546


## Question 2: Text Classification Deep Dive

**Default model:**  
distilbert/distilbert-base-uncased-finetuned-sst-2-english

**Fine-tuned on dataset:**  
SST-2 (Stanford Sentiment Treebank): short movie review sentences labeled positive or negatif.

**What does the score represent?**  
Model confidence (0<x<1).

**Emotion classification model example:**  
"j-hartmann/emotion-english-distilroberta-base"


In [ ]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


,entity_group,score,word,start,end
0,ORG,0.879010,Amazon,5,11
1,MISC,0.990859,Optimus Prime,36,49
2,LOC,0.999755,Germany,90,97
3,MISC,0.556570,Mega,208,212
4,PER,0.590256,##tron,212,216
5,ORG,0.669692,Decept,253,259
6,MISC,0.498349,##icons,259,264
7,MISC,0.775362,Megatron,350,358
8,MISC,0.987854,Optimus Prime,367,380
9,PER,0.812096,Bumblebee,502,511


## Question 3: Named Entity Recognition

**Aggregation_strategy="simple"?**  
It is groupping subword tokens into whole words.

**Entity types:**  
- ORG = Organization  
- PER = Person  
- LOC = Location  
- MISC = Miscellaneous entities  

**'##'?**  
It means that they are subword tokens generated by WordPiece tokenization.

**Why is "Megatron" split incorrectly?**  
Because CoNLL-2003, which primarily consists of news text rather than fictional character names, was used to train the tokenizer.

**What is CoNLL-2003?**  
A dataset of news articles annotated with NER labels


In [ ]:
reader = pipeline("question-answering")
question = "What does the customer want?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


,score,start,end,answer
0,0.631292,335,358,an exchange of Megatron


## Question 4: Question Answering Systems

**Type of QA:**  
Extractive.

**Start/end indices:**  
They locate the predicted answer in the original text.

**SQuAD dataset:**  
Stanford Question Answering Dataset.

**Question the model cannot answer:**  
"Why is Megatron evil?" because it is not in the text.

**Extractive vs generative:**  
- Extractive copies text spans.  
- Generative creates new text.


In [ ]:
summarizer = pipeline("summarization")
outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu
Your min_length=56 must be inferior than your max_length=45.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (45). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


 Bumblebee ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead.


## Question 5: Summarization

**Extractive vs Abstractive:**  
- Extractive keeps original sentences.  
- Abstractive generates new paraphrased text.

**Default model:**  
The default model is sshleifer/distilbart-cnn-12-6. It is an abstractive model that is based on the BART architecture and was trained using the CNN/DailyMail summarization dataset.

**max_length / min_length:**  
Manage the generated summary's length.  
Error/warning if min_length > max_length.

**clean_up_tokenization_spaces=True:**  
Extra spaces are eliminated from decoded text when it's true

**Short-text model example:**  
facebook/bart-large-cnn

**Long-document model example:**  
google/pegasus-large  


In [ ]:
translator = pipeline("translation_en_to_de",
                      model="Helsinki-NLP/opus-mt-en-de")

outputs = translator(text, clean_up_tokenization_spaces=True, min_length=100)
print(outputs[0]['translation_text'])


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Sehr geehrter Amazon, letzte Woche habe ich eine Optimus Prime Action Figur aus Ihrem Online-Shop in Deutschland bestellt. Leider, als ich das Paket öffnete, entdeckte ich zu meinem Entsetzen, dass ich stattdessen eine Action Figur von Megatron geschickt worden war! Als lebenslanger Feind der Decepticons, Ich hoffe, Sie können mein Dilemma verstehen. Um das Problem zu lösen, Ich fordere einen Austausch von Megatron für die Optimus Prime Figur habe ich bestellt. Eingeschlossen sind Kopien meiner Aufzeichnungen über diesen Kauf. Ich erwarte, von Ihnen bald zu hören. Aufrichtig, Bumblebee.


## Question 6: Machine Translation

**Architecture of opus-mt-en-de:**  
MarianMT encoder–decoder Transformer.

**OPUS:**  
Open Parallel Corpus.

**MT:**  
Machine Translation.

**English to French models:**  
- Helsinki-NLP/opus-mt-en-fr  
- facebook/mbart-large-50-many-to-many-mmt

**Bilingual vs multilingual models:**  
- Bilingual = one direction.  
- Multilingual = many languages.

**Sacremoses:**  
Library used for tokenization..

**Example multilingual model:**  
facebook/m2m100-418M (+100 languages).


In [ ]:
from transformers import set_seed
set_seed(42)

generator = pipeline("text-generation")

response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response

outputs = generator(prompt, max_length=200)
print(outputs[0]['generated_text'])


No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee.

Customer service response:
Dear Bumblebee, I am sorry to hear that your order was mixed up. I would like to know if you know more about our service. Please let me know if we can arrange an exchange of Megatron for you.

The following quote from my customer service representative is from my review of the Optimus Prime action figure:

"Hi. I was a bit stunned when I saw the Optimus Prime action figure from your online store. I was hoping you could make me happy, but I was not able to

## Question 7: Text Generation

**Default model:**  
openai-community/gpt2

**Architecture:**  
Decoder-only Transformer.

**Number of parameters:**  
~124M

**Type of generation:**  
Autoregressive.

**Why set_seed?**  
For repeatable outcomes, why set_seed? Without it, different results are produced.

**Generation parameters:**  
Temperature: unpredictability (higher = more inventive)  
Top_k: limits sampling to the top k tokens  
- do_sample: permits sampling rather than avaricious decoding  

**Truncation warning:**  
When input exceeds max_length, it is truncated.

**pad_token_id = eos_token_id:**  
Since GPT-2 lacks a pad token, EOS is utilized.

**Trade-offs:**  
Bigger models are slower, heavier, and have better fluency.
